In [2]:
from pyprojroot import here
import sys
sys.path.insert(0, str(here()))

In [3]:
from src.skyline import SkylineDatabaseGenerator
from src.region import import_region

In [4]:
dem_file = here() / "data" / "digital_elevation_model" / "dem_30m.tif"
region = import_region(here() / "notebooks" / "01_RegionStudy" / "output" / "actual_bounds.json")

In [5]:
import rasterio
with rasterio.open(dem_file) as src:
    print(f"DEM shape: {src.width} x {src.height} = {src.width*src.height:,} pixels")
    print(f"DEM size on disk: {dem_file.stat().st_size / 1e6:.1f} MB")

DEM shape: 34582 x 34216 = 1,183,257,712 pixels
DEM size on disk: 2729.5 MB


In [ ]:
generator = SkylineDatabaseGenerator(
    dem_file=dem_file,
    region=region,
    dist_search_km=30.0
)

In [6]:
db_output_path = here() / "notebooks" / "02_SkylineDatabase" / "output" / "skyline_db.parquet"
mesh_file = here() / "notebooks" / "02_SkylineDatabase" / "output" / "terrain_mesh.npy"
meta_file = here() / "notebooks" / "02_SkylineDatabase" / "output" / "terrain_meta.json"

In [7]:
generator.generate_database(
    output_path=db_output_path,
    grid_spacing_m=30,
    save_raw_horizon=True,
    raw_horizon_stride=1,
    raw_horizon_decimals=1
)

Total candidate viewpoints: 1,336,335
Loading DEM geometry from: /home/admin/SkylineGeolocation/data/digital_elevation_model/dem_30m.tif
Reading cropped DEM window: Window(col_off=16330, row_off=15969, width=3347, height=3007) from source shape (34216, 34582)
--------------------------------------------------------
Horizon computation with Intel Embree
--------------------------------------------------------
DEM dimensions: (3007, 3347) 
Number of vertices: 10064429 
Selected geometry type: grid
BVH build time: 0.299054 s
Total initialisation time: 0.303073 s
Number of locations for which horizon is computed: 4096 
Horizon detection algorithm: binary search
Ray tracing time: 0.851994 s
Number of rays shot: 15267683
Average number of rays per location and azimuth: 10.35 
Total run time: 1.16296 s
--------------------------------------------------------
--------------------------------------------------------
Horizon computation with Intel Embree
-----------------------------------------

'/home/admin/SkylineGeolocation/notebooks/02_SkylineDatabase/output/skyline_db.parquet'

In [7]:
import pyarrow.parquet as pq

parquet_file = pq.ParquetFile(db_output_path)
num_viewpoints = parquet_file.metadata.num_rows
file_size_mb = db_output_path.stat().st_size / (1024 * 1024)

print(f"Total Viewpoints Saved: {num_viewpoints:,}")
print(f"File Size on Disk: {file_size_mb:.2f} MB")

df_head = next(parquet_file.iter_batches(batch_size=5)).to_pandas()
df_head

Total Viewpoints Saved: 1,338,650
File Size on Disk: 910.53 MB


,lon,lat,elevation_m,raw_horizon_deg
0,86.582000,27.77,5163.205078,"[16.0, 15.300000190734863, 15.100000381469727,..."
1,86.582305,27.77,5156.297363,"[15.800000190734863, 15.300000190734863, 15.30..."
2,86.582609,27.77,5160.631836,"[15.5, 15.300000190734863, 15.300000190734863,..."
3,86.582914,27.77,5160.962402,"[15.300000190734863, 15.300000190734863, 15.30..."
4,86.583219,27.77,5157.488281,"[15.300000190734863, 15.300000190734863, 15.30..."
